# **Prova Estrazione Highlights**

In questa prova utilizzo il dataset di HaSpeeDe, Evalita 2018, la cui definizione usata è:

”HS can be defined as any expression that is abusive, insulting,
intimidating, harassing, and/or incites to violence, hatred, or dis-
crimination. It is directed against people on the basis of their race,
ethnic origin, religion, gender, age, physical condition, disability,
sexual orientation, political conviction, and so forth”

 Abuse (aggression), Age (demographics), Appearance (other characteristics), Disability (demographics), Disease (other characteristics), Ethnicity (demographics), Gender (demographics), Hate (aggression), Insult (verbal/written expressions), Political Opinion (beliefs), Race (demographics), Religion (beliefs), Sexual Orientation (demographics), Violate (aggression), discrimination (discrimination/prejudice), harass (aggression), harassing (aggression), insulting (verbal/written expressions), intimidating (sociocultural control), violence (aggression, physical action).

In particolare utilizzo il dataset preso da twitter.

I modelli che utilizzo per le estrazioni sono Llama-3.1-8b-Instruct e Qwen2.5-7B-Instruct

# HF login e imports

In [ ]:
!pip install -U huggingface_hub

In [ ]:
!hf auth login --token hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import numpy as np
import json
import random
from IPython.display import display, HTML

# Data Processing

In [ ]:
!gdown 1nm9A0aBiJTevK7T8QzOmQL8JVB1YRe9b

In [ ]:
df=pd.read_csv('/content/haspeede_TW-train.tsv', sep='\t', header=None)
df.head()

In [ ]:
hate_rows=df[df[2]==1]
texts=hate_rows[1].sample(100)

texts.to_csv('sampled_rows_evalita.csv', header=False)

In [ ]:
texts = texts.reset_index(drop=True)

# Load Model

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name_1="meta-llama/Llama-3.1-8B-Instruct"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(
    model_name_1,
    device_map="auto",
    quantization_config=quantization_config
  )

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name_2="Qwen/Qwen2.5-7B-Instruct"

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(
    model_name_2,
    device_map="auto",
    quantization_config=quantization_config
  )

# Prompt

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse
        - Age
        - Appearence
        - Disability
        - Disease
        - Ethnicity
        - Gender
        - Hate
        - Insult
        - Political Opinion
        - Race
        - Discrimination
        - Religion
        - Sexual Orientation
        - Violence
        - Harrassing
        - Insulting
        - Intimidating


        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be only the raw json object. Do not wrap the json in markdown formatting.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Age": "extracted substring or absent",
            "Appearence": "extracted substring or absent",
            "Disability": "extracted substring or absent",
            "Disease": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Hate": "extracted substring or absent",
            "Insult": "extracted substring or absent",
            "Political Opinion": "extracted substring or absent",
            "Race": "extracted substring or absent",
            "Discrimination": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent",
            "Violence": "extracted substring or absent",
            "Harrassing": "extracted substring or absent",
            "Insulting": "extracted substring or absent",
            "Intimidating": "extracted substring or absent"
        }


        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse: Content which involves improper treatment of a person or group, often to unfairly or improperly gain benefit.
        - Age
        - Appearence
        - Disability
        - Disease
        - Ethnicity: Any reference to a person's or group's ethnic origin.
        - Gender: Any reference to a person's or group's gender identity or biological sex.
        - Hate
        - Insult
        - Political Opinion
        - Race
        - Discrimination
        - Religion: Any reference to a person's or group's religious beliefs, practices, or affiliation.
        - Sexual Orientation: Any reference to a person's or group's sexual orientation.
        - Violence
        - Harrassing
        - Insulting
        - Intimidating


        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be only the raw json object. Do not wrap the json in markdown formatting.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Age": "extracted substring or absent",
            "Appearence": "extracted substring or absent",
            "Disability": "extracted substring or absent",
            "Disease": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Hate": "extracted substring or absent",
            "Insult": "extracted substring or absent",
            "Political Opinion": "extracted substring or absent",
            "Race": "extracted substring or absent",
            "Discrimination": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent",
            "Violence": "extracted substring or absent",
            "Harrassing": "extracted substring or absent",
            "Insulting": "extracted substring or absent",
            "Intimidating": "extracted substring or absent"
        }


        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [ ]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  formatted_prompts=[]

  for text in texts:
    prompt_copy = [
            {"role": prompt_template[0]["role"], "content": prompt_template[0]["content"]},
            {
                "role": prompt_template[1]["role"],
                "content": prompt_template[1]["content"].replace("{text}", text)
            }
        ]


    formatted_prompt = tokenizer.apply_chat_template(
      prompt_copy,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    )

    formatted_prompts.append(formatted_prompt)


  return formatted_prompts

# Inference

In [ ]:
def generate_responses(model, prompt_examples, tokenizer, verbose=False):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      prompt_examples: pre-processed text samples

    Outputs:
      generated responses
  """
  outputs=[]
  for index, prompt_example in enumerate(prompt_examples):
    if verbose and index % 10 == 0:
      print(f"generating response n.{index}...")
    prompt_example=prompt_example.to(model.device)
    output = model.generate(**prompt_example, max_new_tokens=500,pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0][prompt_example["input_ids"].shape[-1]:], skip_special_tokens=True)
    outputs.append(output)

  return outputs

In [ ]:
outputs1 = generate_responses(model_1, prepare_prompts(texts, prompt, tokenizer_1), tokenizer_1, verbose=True)
#40 minuti

In [ ]:
parsed_outputs = []
json_errors = []
for output in outputs1:
    try:
        parsed_outputs.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors.append(output)

with open("results_evalita_1.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs, f, ensure_ascii=False, indent=2)

with open("json_errors_evalita_1.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors))

In [ ]:
outputs2 = generate_responses(model_2, prepare_prompts(texts, prompt, tokenizer_2), tokenizer_2, verbose=True)

In [ ]:
parsed_outputs2 = []
json_errors2 = []
for output in outputs2:
    try:
        parsed_outputs2.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors2.append(output)

with open("results_evalita_2.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs2, f, ensure_ascii=False, indent=2)

with open("json_errors_evalita_2.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors2))

# Extraction Results

In [ ]:
def highlights(texts, outputs):
    """
    This function displays the given text with the model extracted highlights.
    """
    colori = {
        "Abuse":              "#FFB3B3",  # rosso chiaro
        "Age":                "#FFE4B3",  # arancio chiaro
        "Appearence":         "#FFF3B3",  # giallo chiaro
        "Disability":         "#B3FFE4",  # verde acqua
        "Disease":            "#B3F0FF",  # azzurro chiaro
        "Ethnicity":          "#B3D9FF",  # blu chiaro
        "Gender":             "#E6B3FF",  # viola chiaro
        "Hate":               "#FF9999",  # rosso
        "Insult":             "#FFB3D9",  # rosa
        "Political Opinion":  "#C8FFB3",  # verde chiaro
        "Race":               "#B3FFFF",  # ciano chiaro
        "Discrimination":     "#FFD9B3",  # pesca
        "Religion":           "#B3FFB3",  # verde
        "Sexual Orientation": "#F0B3FF",  # lilla
        "Violence":           "#FF8080",  # rosso scuro
        "Harrassing":         "#FFB3B3",  # rosso chiaro 2
        "Insulting":          "#FFD4B3",  # albicocca
        "Intimidating":       "#D4B3FF",  # lavanda
    }

    testo_colorato = texts
    for categoria, estratto in outputs.items():
        if estratto and estratto.lower() not in ("null", "absent") and estratto in testo_colorato:
            colore = colori.get(categoria, "#DDDDDD")
            tag_html = (
                f"<mark style='background-color: {colore}; padding: 2px 4px; border-radius: 4px;'>"
                f"{estratto} "
                f"<span style='font-size: 0.7em; font-weight: bold;'>[{categoria}]</span>"
                f"</mark>"
            )
            testo_colorato = testo_colorato.replace(estratto, tag_html)

    display(HTML(f"<p style='line-height: 1.8; font-size: 16px;'>{testo_colorato}</p>"))


In [ ]:
for i in range(len(outputs1)):
    parsed_output = {}
    try:
        parsed_output = json.loads(outputs1[i])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e} for text: {texts[i]}, model output: {outputs1[i]}")

    highlights(texts[i], parsed_output)

In [ ]:
for i in range(len(outputs2)):
    parsed_output = {}
    try:
        parsed_output = json.loads(outputs2[i])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e} for text: {texts[i]}, model output: {outputs2[i]}")

    highlights(texts[i], parsed_output)